# A Connes Noncommutative-Geometry Pipeline for Topological Invariants

This notebook implements topological-invariant extraction as a **discrete spectral-triple / cyclic-cohomology** computation — Connes' noncommutative geometry, the rigorous framework behind Bellissard's theory of the quantum Hall effect. The pipeline (10 stages):

1. **Spectral triple** $(\mathcal{A},\mathcal{H},D)$: operator algebra, Hilbert space, a discrete Dirac operator $D$.
2. **Differential** $dO_i := [D,O_i]$ — the discrete analogue of $df$. Geometry lives in operator *variation*.
3. **Relational cochains** $\omega_{ij} = \mathrm{Tr}(P\,[D,O_i][D,O_j])$ — a **complex** trace that keeps order, phase, antisymmetry.
4. **Connection** $U_{ij} = \omega_{ij}/|\omega_{ij}| \in U(1)$ — discrete gauge links.
5. **Simplicial dual geometry** on operator sectors (vertices), influence transport (edges), cyclic loops (triangles).
6. **Curvature** $F_{ijk} = \arg(U_{ij}U_{jk}U_{ki})$ — discrete Berry curvature / simplicial holonomy.
7. **Cyclic pairing** $\phi(a_0,a_1,a_2) = \mathrm{Tr}(a_0[D,a_1][D,a_2])$ — the Connes–Chern character.
8. **Invariant** $\langle\phi,[P]\rangle = \phi(P,P,P)$ → Chern / QSH / winding numbers.
9. **Persistence**: which cocycles survive noise (robustness of the influence geometry).
10. **Sector resolution**: charge→Chern, spin→QSH, Floquet→anomalous winding, etc.

**Three corrections** that make it actually run, learned the hard way:
- **$D$ must carry two directions** (a Clifford/two-derivation structure $D\sim X\sigma_1+Y\sigma_2$); a single scalar $D$ gives no curvature.
- **On a torus, positions must be exponentiated** (twist / magnetic-translation operators) — bare $X$ isn't periodic, and that non-periodicity is exactly what made earlier many-body attempts eject the ground manifold.
- **Read the invariant from the direct pairing $\phi(P,P,P)$** (Stage 8), not the normalized-edge layer (Stage 4 discards $|\omega_{ij}|$).

The decisive feature vs. a commutator-*norm* filtration: $\omega_{ij}$ is a **phase-carrying** complex trace. A norm $\lVert[O_i,O_j]\rVert$ discards exactly the phase the invariant is made of.

## Test 1 — Single particle: Chern number of a lattice insulator (grid-free)

Spectral triple for the QWZ model: $\mathcal{A}$ = single-particle observables, $\mathcal{H}$ = lattice Hilbert space, $D$ built from the **exponentiated** positions $U_x=e^{2\pi i X/L_x}$, $U_y=e^{2\pi i Y/L_y}$ (torus-correct). The Connes–Chern pairing $\phi(P,P,P)$ with $P$ the occupied projector is computed via the gauge-invariant log-trace (the Loring–Bellissard form). No Brillouin zone, no flux grid.

In [12]:
import numpy as np, scipy.linalg as la
from itertools import combinations, product
sx=np.array([[0,1],[1,0]],complex); sy=np.array([[0,-1j],[1j,0]]); sz=np.array([[1,0],[0,-1]],complex)

def qwz(Lx,Ly,M):
    idx=lambda x,y,o:2*((x%Lx)*Ly+(y%Ly))+o; N=2*Lx*Ly; H=np.zeros((N,N),complex); Tx=(-sz-1j*sx)/2; Ty=(-sz-1j*sy)/2
    for x,y in product(range(Lx),range(Ly)):
        for a in(0,1):
            for b in(0,1): H[idx(x,y,a),idx(x,y,b)]+=M*sz[a,b]
        for(dx,dy,T)in[(1,0,Tx),(0,1,Ty)]:
            xn,yn=(x+dx)%Lx,(y+dy)%Ly
            for a in(0,1):
                for b in(0,1): H[idx(x,y,a),idx(xn,yn,b)]+=T[a,b]; H[idx(xn,yn,b),idx(x,y,a)]+=np.conj(T[a,b])
    return H

def connes_chern_sp(Lx,Ly,M):
    """Stage 1-8 for one particle: P=occupied projector, D=exponentiated positions, phi(P,P,P) as log-trace."""
    H=qwz(Lx,Ly,M); e,V=la.eigh(H); Nocc=Lx*Ly; Psi=V[:,:Nocc]            # Stage 1: (A,H,D), P
    xs=np.array([(s//2)//Ly for s in range(2*Lx*Ly)]); ys=np.array([(s//2)%Ly for s in range(2*Lx*Ly)])
    Ux=np.exp(2j*np.pi*xs/Lx); Uy=np.exp(2j*np.pi*ys/Ly)                   # Stage 2: exponentiated D
    ux=Psi.conj().T@(Ux[:,None]*Psi); uy=Psi.conj().T@(Uy[:,None]*Psi)    # Stage 3: P-projected transport
    M_=ux@uy@la.inv(ux)@la.inv(uy)                                        # Stage 6-7: cyclic curvature product
    return (1/(2*np.pi))*np.imag(np.sum(np.log(la.eigvals(M_))))          # Stage 8: <phi,[P]>

print("Connes-Chern pairing  <phi,[P]>  (single particle, grid-free):")
for nm,M in [("Topological (M=1)",1.0),("Trivial (M=3)",3.0)]:
    print(f"   QWZ {nm}: C = {connes_chern_sp(4,3,M):+.4f}")
print("\n=> clean integers, no Brillouin-zone / flux grid. The pipeline computes a real Chern number.")

Connes-Chern pairing  <phi,[P]>  (single particle, grid-free):
   QWZ Topological (M=1): C = +1.0000
   QWZ Trivial (M=3): C = +0.0000

=> clean integers, no Brillouin-zone / flux grid. The pipeline computes a real Chern number.


## Test 2 — Many body: fractional Chern of the $\nu=1/3$ Laughlin state (grid-free)

The hard case. We use the lowest Landau level on a torus ($N_\phi=9$ orbitals, $N=3$ fermions) — a genuinely **ideal** band (uniform curvature), unlike any small lattice model. 

**Correctness gate first:** the $V_1$ Haldane pseudopotential must give the exact 3-fold zero-energy Laughlin manifold. Only then do we trust the invariant.

Then the spectral triple is many-body: $\mathcal{H}$ = Fock space, $P$ = projector onto the 3-fold manifold, and $D$ = the **magnetic translations** $T_1$ (orbital shift) and $T_2$ (phase) — the exponentiated guiding-center operators. The Connes–Chern pairing $\phi(P,P,P)$ via the same log-trace gives the **total** Chern of the manifold; dividing by $q=3$ gives the fractional value.

In [13]:
def build_V(Nphi,smax=6):
    L=np.sqrt(2*np.pi*Nphi)
    def Vt(qx,qy): q2=qx*qx+qy*qy; return (1-q2)*np.exp(-q2/2)   # V_1 pseudopotential (Laughlin parent)
    V=np.zeros((Nphi,)*4,complex)
    for j1 in range(Nphi):
        for j2 in range(Nphi):
            for j3 in range(Nphi):
                j4=(j1+j2-j3)%Nphi; m=(j1-j3)%Nphi; acc=0j
                for s in range(-smax,smax+1):
                    qx=2*np.pi*s/L
                    for l in range(-smax,smax+1):
                        t=m+Nphi*l; qy=2*np.pi*t/L
                        acc+=Vt(qx,qy)*np.exp(2j*np.pi*s*(j1-j3)/Nphi)*np.exp(-1j*np.pi*s*t/Nphi)
                V[j1,j2,j3,j4]=acc/Nphi
    return V

def mbH(Nphi,N,V):
    states=list(combinations(range(Nphi),N)); idx={s:i for i,s in enumerate(states)}; D=len(states)
    H=np.zeros((D,D),complex)
    for si,occ in enumerate(states):
        oset=set(occ)
        for a in range(Nphi):
            for b in range(Nphi):
                if a==b: continue
                for c in range(Nphi):
                    d=(a+b-c)%Nphi
                    if d==c or c not in oset or d not in oset: continue
                    o2=list(occ); s1=(-1)**o2.index(d); o2.remove(d)
                    if c not in o2: continue
                    s2=(-1)**o2.index(c); o2.remove(c)
                    if a in o2 or b in o2: continue
                    tmp=sorted(o2+[b]); s3=(-1)**tmp.index(b)
                    if a in tmp: continue
                    fin=sorted(tmp+[a]); s4=(-1)**fin.index(a)
                    H[idx[tuple(fin)],si]+=0.5*V[a,b,c,d]*s1*s2*s3*s4
    return 0.5*(H+H.conj().T),states,idx

Nphi,N=12,4
V=build_V(Nphi); H,states,idx=mbH(Nphi,N,V); Dh=len(states)
w,U=la.eigh(H); sp=w-w[0]
print(f"CORRECTNESS GATE  (LLL V1, Nphi={Nphi}, N={N}, nu=1/3):")
print(f"   lowest 6 energies = {np.round(sp[:6],5)}")
print(f"   3 zero modes + gap {sp[3]-sp[2]:.4f}  => exact Laughlin manifold confirmed.\n")
G=U[:,:3]   # P = projector onto the 3-fold Laughlin manifold

CORRECTNESS GATE  (LLL V1, Nphi=12, N=4, nu=1/3):
   lowest 6 energies = [0.      0.      0.      0.14851 0.14851 0.14851]
   3 zero modes + gap 0.1485  => exact Laughlin manifold confirmed.



In [14]:
# Stage 2: many-body exponentiated D = magnetic translations T1 (orbital shift), T2 (phase)
def T1_mat():
    Mt=np.zeros((Dh,Dh),complex)
    for si,occ in enumerate(states):
        new=[(j+1)%Nphi for j in occ]
        inv=sum(1 for a in range(N) for b in range(a+1,N) if new[a]>new[b])  # fermion sign
        Mt[idx[tuple(sorted(new))],si]=(-1)**inv
    return Mt
def T2_mat():
    return np.diag([np.exp(2j*np.pi*sum(occ)/Nphi) for occ in states])
T1=T1_mat(); T2=T2_mat()

def connes_chern_mb(Gsub):
    """Stage 3-8: project D-transport onto manifold P, cyclic curvature product, pairing."""
    u1=Gsub.conj().T@T1@Gsub; u2=Gsub.conj().T@T2@Gsub
    defect=max(np.abs(u1@u1.conj().T-np.eye(Gsub.shape[1])).max(),
               np.abs(u2@u2.conj().T-np.eye(Gsub.shape[1])).max())
    Mc=u1@u2@la.inv(u1)@la.inv(u2)
    K=(1/(2*np.pi))*np.imag(np.sum(np.log(la.eigvals(Mc))))
    return K, defect

K,defect=connes_chern_mb(G)
print(f"LAUGHLIN manifold:  <phi,[P]> = K = {K:+.4f}  ->  K/q = {K/3:+.4f}")
print(f"   projected-D unitarity defect = {defect:.4f}  (0 => manifold preserved by D, i.e. ideal band)")
# Control: a random 3-dim subspace is NOT a topological manifold
rng=np.random.default_rng(0); R=la.qr(rng.normal(size=(Dh,3))+1j*rng.normal(size=(Dh,3)),mode='economic')[0]
Kr,defr=connes_chern_mb(R)
print(f"\nRANDOM 3-subspace control:  K = {Kr:+.4f}   unitarity defect = {defr:.4f}")
print("   => the clean fractional invariant is specific to the topological ground manifold, not generic.")

LAUGHLIN manifold:  <phi,[P]> = K = -1.0000  ->  K/q = -0.3333
   projected-D unitarity defect = 0.0000  (0 => manifold preserved by D, i.e. ideal band)

RANDOM 3-subspace control:  K = +0.0000   unitarity defect = 0.9971
   => the clean fractional invariant is specific to the topological ground manifold, not generic.


In [27]:
def anderson_sweep_unfiltered(Wlist, nsamp=50, corr_steps=2, seed=0):
    """
    Sweeps through disorder strengths using correlated fields, computing the 
    topological invariant unconditionally for every single sample.
    """
    rng = np.random.default_rng(seed)
    out = []
    
    # Bind to your notebook's existing background quantum states and operators
    global H, states, idx, Nphi
    
    for W in Wlist:
        Ks = []
        defects = []
        splits = []
        gaps = []
        
        for _ in range(nsamp):
            # Generate a unique seed per disorder configuration
            sample_seed = int(rng.integers(0, 2**32 - 1))
            disorder = correlated_disorder_field(W, Nphi=Nphi, corr_steps=corr_steps, seed=sample_seed)
            
            # Construct the disordered Hamiltonian and diagonalize
            Hd = H + np.diag(anderson_diag(disorder))
            ev, U = la.eigh(Hd)
            
            # Pull the 3-fold lowest many-body states
            Gsub = U[:, :3]
            
            # Unconditionally run the Connes-Chern pairing trace loop
            K, defect = connes_chern_mb(Gsub)
            
            split = float(ev[2] - ev[0])
            gap = float(ev[3] - ev[2])
            
            Ks.append(K)
            defects.append(defect)
            splits.append(split)
            gaps.append(gap)
            
        Ks = np.array(Ks, float)
        defects = np.array(defects, float)
        splits = np.array(splits, float)
        gaps = np.array(gaps, float)
        
        out.append((W, Ks.mean(), Ks.std(), defects.mean(), splits.mean(), gaps.mean()))
        
        # Cleaner printout without any filter metrics
        print(f"W={W:.3f}  <K>={Ks.mean():+.6f}±{Ks.std():.6f}  <defect>={defects.mean():.6f}  "
              f"<split>={splits.mean():.6f}  <gap>={gaps.mean():.6f}")
            
    return np.array(out, dtype=float)

# Execute the unfiltered pipeline
Wlist = np.linspace(0.0, 0.20, 7)
res = anderson_sweep_unfiltered(Wlist, nsamp=50, corr_steps=2, seed=1)

W=0.000  <K>=-1.000000±0.000000  <defect>=0.000000  <split>=0.000000  <gap>=0.148511
W=0.033  <K>=-1.000000±0.000000  <defect>=0.013770  <split>=0.000282  <gap>=0.126176
W=0.067  <K>=-1.000000±0.000000  <defect>=0.081791  <split>=0.003609  <gap>=0.103494
W=0.100  <K>=-1.000000±0.000000  <defect>=0.225307  <split>=0.012135  <gap>=0.087253
W=0.133  <K>=-1.000000±0.000000  <defect>=0.404007  <split>=0.025829  <gap>=0.072402
W=0.167  <K>=-0.960000±0.195959  <defect>=0.600830  <split>=0.055061  <gap>=0.049584
W=0.200  <K>=-0.940000±0.237487  <defect>=0.748573  <split>=0.073314  <gap>=0.042418
